# End-to-end workflow

In [ ]:
import kagglehub

kagglehub.login()
# Kaggle token removed. Use kagglehub.login() or Kaggle Secrets.

In [ ]:
# Dataset path setup.
# In Colab/Kaggle, the paths may come from KaggleHub variables created by earlier cells.
# For local execution, place datasets under ./data as documented in data/README.md,
# or replace the fallback paths below with your own local dataset directories.
# Example local variables:
# happy_whale_and_dolphin_path = "data/happy-whale-and-dolphin"
# jpbremer_fullbodywhaleannotations_path = "data/fullbodywhaleannotations"
# backfin_path = "data/happy-whale-backfin-cropped"


def _get_or_download_kaggle_paths():
    official_path = globals().get("happy_whale_and_dolphin_path", None)
    fullbody_path = globals().get(
        "jpbremer_fullbodywhaleannotations_path", None
    )
    backfin_path_local = globals().get("backfin_path", None)

    try:
        if official_path is None or not os.path.exists(str(official_path)):
            official_path = "data/happy-whale-and-dolphin"
        if fullbody_path is None or not os.path.exists(str(fullbody_path)):
            fullbody_path = "data/fullbodywhaleannotations"
        if backfin_path_local is None or not os.path.exists(
            str(backfin_path_local)
        ):
            backfin_path_local = "data/happy-whale-backfin-cropped"
    except Exception as e:
        print(
            "KaggleHub auto-download fallback failed. Will try /kaggle/input candidates instead."
        )
        print("Error:", e)

    return official_path, fullbody_path, backfin_path_local


KAGGLEHUB_OFFICIAL_DIR, KAGGLEHUB_FULLBODY_DIR, KAGGLEHUB_BACKFIN_DIR = (
    _get_or_download_kaggle_paths()
)

In [ ]:
# ============================================================
# Import packages
# ============================================================

!pip install -q timm albumentations huggingface_hub

import ast
import gc
import math
import os
import random
import re
import shutil
import time
import warnings
from pathlib import Path

import cv2

warnings.filterwarnings("ignore")

import albumentations as A
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from huggingface_hub import HfApi, create_repo, login
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

In [ ]:
# ============================================================
# Config
# ============================================================


class CFG:
    # ============================================================
    # 0. Basic
    # ============================================================
    seed = 42
    device = "cuda" if torch.cuda.is_available() else "cpu"

    exp_name = "happywhale_v2xl_pseudo_768"
    output_dir = os.path.join(exp_name)
    os.makedirs(output_dir, exist_ok=True)

    # ============================================================
    # 1. Run Mode
    # ============================================================
    # This setting controls what this notebook should run.
    # Training notebooks can use: "train", "extract_embeddings", "train_then_extract".
    # Inference notebooks can use: "crop_inference", "download_embeddings", "build_mats".
    run_mode = "crop_inference"

    do_train = True
    do_embedding = False
    do_crop_inference = False

    # ============================================================
    # 2. Data Paths
    # ============================================================
    official_data_candidates = [
        KAGGLEHUB_OFFICIAL_DIR,
        "/content/happy-whale-and-dolphin",
        "/content/competitions/happy-whale-and-dolphin",
        "/kaggle/input/happy-whale-and-dolphin",
    ]
    official_data_candidates = [
        str(d) for d in official_data_candidates if d is not None
    ]

    data_dir = next(
        (
            d
            for d in official_data_candidates
            if os.path.exists(os.path.join(d, "train.csv"))
        ),
        None,
    )

    fullbody_dir = (
        str(KAGGLEHUB_FULLBODY_DIR)
        if KAGGLEHUB_FULLBODY_DIR is not None
        else None
    )
    backfin_dir = (
        str(KAGGLEHUB_BACKFIN_DIR)
        if KAGGLEHUB_BACKFIN_DIR is not None
        else None
    )

    # ============================================================
    # 3. Hugging Face
    # ============================================================
    hf_upload = True
    hf_repo_id = os.environ.get(
        "HF_REPO_ID",
        "fangfang777/happywhale_v2xl_pseudo_768",
    )
    hf_repo_type = "model"

    hf_embedding_dir = "embeddings"
    hf_checkpoint_dir = "checkpoints"
    hf_mat_dir = "mats"
    hf_submission_dir = "submissions"

    hf_upload_each_epoch = True
    hf_upload_each_five_epoch = True

    # ============================================================
    # 4. Colab / Runtime
    # ============================================================
    auto_disconnect_when_done = False

    # ============================================================
    # 5. Model Architecture
    # ============================================================
    model_name = "tf_efficientnetv2_xl.in21k"
    image_size = 768

    arc_s = 30.0
    arc_m = 0.30

    # Sub-center ArcFace
    n_center_id = 3
    n_center_species = 3

    # Adaptive margin
    s_id = 30.0
    s_species = 30.0

    margin_power_id = -0.25
    margin_power_species = -0.25
    margin_coef_id = 0.40
    margin_coef_species = 0.40
    margin_cons_id = 0.0
    margin_cons_species = 0.0

    # ============================================================
    # 6. Training
    # ============================================================
    resume = True
    hf_checkpoint_file = "checkpoints/last_checkpoint.pth"  # Checkpoint path to download from Hugging Face.

    batch_size = 4
    grad_accum_steps = 2
    grad_clip = 1.0

    num_workers = 4
    epochs = 75

    lr_backbone = 3e-5
    lr_head = 7e-4
    weight_decay = 1e-4

    use_fullbody_crop = True
    use_backfin_crop = True
    crop_margin = 0.05

    crop_probs = {
        "fullbody": 0.70,
        "backfin": 0.20,
        "none": 0.10,
    }

    # Species auxiliary head + individual head loss ratio
    loss_id_ratio = 0.9

    # ============================================================
    # 7. Focal Loss
    # ============================================================
    use_focal_loss = True
    focal_gamma_id = 1.25
    focal_gamma_species = 1.0

    use_focal_alpha = False
    focal_alpha_beta_id = 0.999
    focal_alpha_beta_species = 0.99

    # ============================================================
    # 8. Pseudo-label Training
    # ============================================================
    use_pseudo = True
    pseudo_csv = os.environ.get("PSEUDO_CSV", "data/pseudo.csv")

    pseudo_top1_threshold = 0.75
    pseudo_margin_threshold = 0.12
    pseudo_max_per_id = 15
    pseudo_drop_new_individual = True

    # ============================================================
    # 9. Embedding Extraction
    # ============================================================
    infer_batch_size = 64

    # Crop modes used when exporting embeddings.
    crop_infer_modes = ["fullbody", "backfin", "none"]

    # ============================================================
    # 10. Inference: KNN / Logit
    # ============================================================
    knn_neighbors = 500
    knn_chunk_size = 128

    logits_topk = 1000

    # Used by the legacy KNN + logit inference path.
    knn_ratio = 0.8

    # new_individual ratio search
    new_ratios = [0.15, 0.165, 0.20, 0.215]

    # ============================================================
    # 11. Inference: Prototype
    # ============================================================
    use_proto_score = True

    # "single" or "multi"
    proto_mode = "single"

    # KNN / logit / prototype fusion weights
    knn_weight = 0.50
    logit_weight = 0.25
    proto_weight = 0.25

    # multi-prototype settings
    multi_proto_max_centers = 3
    multi_proto_min_samples = 4

    # ============================================================
    # 12. Inference: Species Bonus
    # ============================================================
    use_species_bonus = True
    species_topk = 30
    species_use_topk = 1

    species_bonus = 0.10
    species_penalty = 0.00

    # ============================================================
    # 13. Crop Ensemble
    # ============================================================
    crop_weight_sets = [
        {
            "name": "fb080_bf010_none010",
            "weights": {"fullbody": 0.80, "backfin": 0.10, "none": 0.10},
        },
        {
            "name": "fb085_bf010_none005",
            "weights": {"fullbody": 0.85, "backfin": 0.10, "none": 0.05},
        },
        {
            "name": "fb085_bf005_none010",
            "weights": {"fullbody": 0.85, "backfin": 0.05, "none": 0.10},
        },
        {
            "name": "fb090_bf005_none005",
            "weights": {"fullbody": 0.90, "backfin": 0.05, "none": 0.05},
        },
        {
            "name": "fb090_bf010_none000",
            "weights": {"fullbody": 0.90, "backfin": 0.10, "none": 0.00},
        },
        {
            "name": "fb095_bf003_none002",
            "weights": {"fullbody": 0.95, "backfin": 0.03, "none": 0.02},
        },
    ]

    # ============================================================
    # 14. Output Names for Crop Inference
    # ============================================================

    if use_proto_score:
        # New version: KNN + logits + prototypes.
        score_tag = (
            f"knn{int(knn_weight * 100):02d}"
            f"_logit{int(logit_weight * 100):02d}"
            f"_proto{int(proto_weight * 100):02d}"
        )
        proto_tag = f"proto_{proto_mode}"

    else:
        # Legacy version: KNN + logits only.
        score_tag = f"knnratio{int(knn_ratio * 100):02d}"
        proto_tag = "no_proto"

    species_tag = (
        f"bonus{species_bonus:.2f}_penalty{species_penalty:.2f}"
        if use_species_bonus
        else "no_species"
    )

    # Fixed component-matrix cache name. Do not include ensemble weights here.
    component_mat_dir = "crop_component_mats"

    # Submission directory includes the weight configuration.
    crop_submission_dir = (
        f"crop_submission_{proto_tag}_{score_tag}_{species_tag}"
    )

## HF function

In [ ]:
# ============================================================
# Hugging Face helpers
# ============================================================

# def get_hf_token():
#     token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
#     if token:
#         return token

#     # Kaggle Notebook Secret: Add-ons -> Secrets -> HF_TOKEN
#     try:
#         from kaggle_secrets import UserSecretsClient
#         token = UserSecretsClient().get_secret("HF_TOKEN")
#         if token:
#             os.environ["HF_TOKEN"] = token
#             return token
#     except Exception:
#         pass

#     return None


def get_hf_token():
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")

    if not token:
        try:
            from google.colab import userdata

            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None

    if token:
        login(token=token)
        return token

    # If HF_TOKEN is not configured in Colab userdata, paste the token manually.
    from getpass import getpass

    token = getpass("Paste your Hugging Face access token: ")
    login(token=token)

    return token


HF_API = None


def init_hf():
    global HF_API

    if not CFG.hf_upload:
        print("HF upload disabled.")
        return None

    token = get_hf_token()
    if not token:
        raise RuntimeError(
            "HF upload is enabled but no token was found. "
            "On Kaggle, add a Secret named HF_TOKEN, or set os.environ['HF_TOKEN']."
        )

    if not CFG.hf_repo_id or "YOUR_USERNAME" in CFG.hf_repo_id:
        raise RuntimeError(
            "Please set CFG.hf_repo_id to your Hugging Face repo id, "
            "for example: 'your_name/happywhale_b6_multicrop'."
        )

    login(token=token, add_to_git_credential=False)
    create_repo(
        repo_id=CFG.hf_repo_id,
        repo_type=CFG.hf_repo_type,
        private=True,
        exist_ok=True,
        token=token,
    )
    HF_API = HfApi(token=token)
    print("HF repo ready:", CFG.hf_repo_id)
    return HF_API


def hf_upload_file_if_exists(local_path, repo_path=None, commit_message=None):
    if not CFG.hf_upload:
        return
    if HF_API is None:
        return
    if not os.path.exists(local_path):
        print("Skip HF upload, file not found:", local_path)
        return

    repo_path = repo_path or os.path.relpath(local_path, CFG.output_dir)
    commit_message = commit_message or f"Upload {repo_path}"

    HF_API.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=repo_path,
        repo_id=CFG.hf_repo_id,
        repo_type=CFG.hf_repo_type,
        commit_message=commit_message,
    )
    print("Uploaded to HF:", repo_path)


def hf_upload_folder_if_exists(local_dir, repo_path=None, commit_message=None):
    if not CFG.hf_upload:
        return
    if HF_API is None:
        return
    if not os.path.exists(local_dir):
        print("Skip HF upload, folder not found:", local_dir)
        return

    repo_path = repo_path or os.path.relpath(local_dir, CFG.output_dir)
    commit_message = commit_message or f"Upload folder {repo_path}"

    HF_API.upload_folder(
        folder_path=local_dir,
        path_in_repo=repo_path,
        repo_id=CFG.hf_repo_id,
        repo_type=CFG.hf_repo_type,
        commit_message=commit_message,
    )
    print("Uploaded folder to HF:", repo_path)


init_hf()

In [ ]:
def hf_download_embedding_file(repo_path, local_path):
    """
    Download one embedding file from Hugging Face.
    Return True on success and False on failure.
    """
    try:
        from huggingface_hub import hf_hub_download

        token = get_hf_token()

        os.makedirs(os.path.dirname(local_path), exist_ok=True)

        downloaded_path = hf_hub_download(
            repo_id=CFG.hf_repo_id,
            repo_type=CFG.hf_repo_type,
            filename=repo_path,
            token=token,
        )

        # Copy the file to the path expected by the original notebook:
        # CFG.output_dir/train_xxx_results.npz
        import shutil

        shutil.copy2(downloaded_path, local_path)

        print("Downloaded embedding from HF:")
        print(" ", repo_path)
        print(" ->", local_path)

        return True

    except Exception as e:
        print("Failed to download embedding from HF:")
        print(" ", repo_path)
        print(type(e).__name__, str(e))
        return False


def download_embeddings_for_mode(crop_mode):
    """
    Used by do_infer:
    Download train/test npz files from Hugging Face to CFG.output_dir.
    If the files are missing on Hugging Face, remind the user to run with do_embedding=True first.
    """
    train_npz_path = os.path.join(
        CFG.output_dir, f"train_{crop_mode}_results.npz"
    )
    test_npz_path = os.path.join(
        CFG.output_dir, f"test_{crop_mode}_results.npz"
    )

    train_repo_path = f"{CFG.hf_embedding_dir}/train_{crop_mode}_results.npz"
    test_repo_path = f"{CFG.hf_embedding_dir}/test_{crop_mode}_results.npz"

    print(f"\n===== Download embeddings for crop mode: {crop_mode} =====")

    train_ok = hf_download_embedding_file(
        repo_path=train_repo_path,
        local_path=train_npz_path,
    )

    test_ok = hf_download_embedding_file(
        repo_path=test_repo_path,
        local_path=test_npz_path,
    )

    if not train_ok or not test_ok:
        raise FileNotFoundError(
            f"\n[{crop_mode}] Embedding was not found on Hugging Face.\n"
            f"Please set:\n"
            f"  CFG.do_embedding = True\n"
            f"  CFG.do_infer = False or True\n"
            f"Regenerate embeddings and upload them to Hugging Face.\n\n"
            f"HF expected files:\n"
            f"  {train_repo_path}\n"
            f"  {test_repo_path}"
        )

    return train_npz_path, test_npz_path


def load_champion_outputs_for_mode(crop_mode):
    """
    Load local npz files. do_infer downloads them from Hugging Face first, then reads them here.
    """
    train_npz_path = os.path.join(
        CFG.output_dir, f"train_{crop_mode}_results.npz"
    )
    test_npz_path = os.path.join(
        CFG.output_dir, f"test_{crop_mode}_results.npz"
    )

    if not os.path.exists(train_npz_path):
        raise FileNotFoundError(f"Missing local train npz: {train_npz_path}")

    if not os.path.exists(test_npz_path):
        raise FileNotFoundError(f"Missing local test npz: {test_npz_path}")

    print(f"Loading local embeddings for crop mode: {crop_mode}")
    print("Train:", train_npz_path)
    print("Test:", test_npz_path)

    train_results = np.load(train_npz_path, allow_pickle=True)
    test_results = np.load(test_npz_path, allow_pickle=True)

    return train_results, test_results

## Preparation (Required for Training and Inference)
1. Set the random seed for reproducibility.
2. Verify that all dataset paths are correct.
3. Locate the dataset split files.

In [ ]:
# ============================================================
# Seed + device check
# ============================================================


def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(CFG.seed)

print("OUTPUT_DIR:", CFG.output_dir)
print("Device:", CFG.device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# Prepare Happywhale official data
# ============================================================

if CFG.data_dir is None:
    raise FileNotFoundError(
        "Cannot find official Happywhale data under /kaggle/input. "
        "Please add Kaggle competition data: happy-whale-and-dolphin."
    )

TRAIN_CSV = os.path.join(CFG.data_dir, "train.csv")
SAMPLE_SUB_CSV = os.path.join(CFG.data_dir, "sample_submission.csv")
TRAIN_IMG_DIR = os.path.join(CFG.data_dir, "train_images")
TEST_IMG_DIR = os.path.join(CFG.data_dir, "test_images")

required_paths = [TRAIN_CSV, SAMPLE_SUB_CSV, TRAIN_IMG_DIR, TEST_IMG_DIR]
missing = [p for p in required_paths if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        "Missing required Kaggle input paths: " + str(missing)
    )

print("DATA_DIR:", CFG.data_dir)
print("TRAIN_CSV:", TRAIN_CSV)
print("SAMPLE_SUB_CSV:", SAMPLE_SUB_CSV)
print("TRAIN_IMG_DIR:", TRAIN_IMG_DIR)
print("TEST_IMG_DIR:", TEST_IMG_DIR)
print("OUTPUT_DIR:", CFG.output_dir)

In [ ]:
# ============================================================
# Download / locate fullbody annotation dataset
# ============================================================

FULLBODY_DIR = (
    CFG.fullbody_dir
    if CFG.fullbody_dir and os.path.exists(CFG.fullbody_dir)
    else None
)
candidate_fullbody_dirs = [
    "/kaggle/input/fullbodywhaleannotations",
    "/kaggle/input/fullbody-whale-annotations",
    "/kaggle/input/jpbremer-fullbodywhaleannotations",
    "/content/fullbodywhaleannotations",
]

if FULLBODY_DIR is None:
    for d in candidate_fullbody_dirs:
        if os.path.exists(d):
            FULLBODY_DIR = d
            break

if FULLBODY_DIR is None:
    try:
        import kagglehub

        print("Trying to download fullbody annotations...")
        FULLBODY_DIR = kagglehub.dataset_download(
            "jpbremer/fullbodywhaleannotations"
        )
    except Exception as e:
        print("Could not download fullbody annotations.")
        print("Error:", e)
        CFG.use_fullbody_crop = False

print("FULLBODY_DIR:", FULLBODY_DIR)

In [ ]:
# ============================================================
# Download / locate backfin cropped dataset
# ============================================================

BACKFIN_DIR = (
    CFG.backfin_dir
    if CFG.backfin_dir and os.path.exists(CFG.backfin_dir)
    else None
)
candidate_backfin_dirs = [
    "/kaggle/input/happy-whale-backfin-cropped",
    "/kaggle/input/happy-whale-backfin-crop",
    "/kaggle/input/genlaxai-happy-whale-backfin-cropped",
    "/content/happy-whale-backfin-cropped",
]

if BACKFIN_DIR is None:
    for d in candidate_backfin_dirs:
        if os.path.exists(d):
            BACKFIN_DIR = d
            break

if BACKFIN_DIR is None:
    try:
        import kagglehub

        print("Trying to download backfin cropped dataset...")
        BACKFIN_DIR = kagglehub.dataset_download(
            "genlaxai/happy-whale-backfin-cropped"
        )
    except Exception as e:
        print("Could not download backfin dataset.")
        print("Error:", e)
        CFG.use_backfin_crop = False

print("BACKFIN_DIR:", BACKFIN_DIR)

In [ ]:
# ============================================================
# Utility: CSV and bbox parsing
# ============================================================


def find_csv_files(root_dir):
    csvs = []
    if root_dir is None:
        return csvs
    for root, dirs, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith(".csv"):
                csvs.append(os.path.join(root, f))
    return csvs


def find_image_folders(root_dir):
    folders = []
    if root_dir is None:
        return folders

    for root, dirs, files in os.walk(root_dir):
        image_count = sum(
            f.lower().endswith((".jpg", ".jpeg", ".png")) for f in files
        )
        if image_count > 10:
            folders.append(root)

    return folders


def choose_train_test_csv(csv_files, train_key="train", test_key="test"):
    train_csv = None
    test_csv = None

    for p in csv_files:
        name = os.path.basename(p).lower()
        if train_key in name and train_csv is None:
            train_csv = p
        if test_key in name and test_csv is None:
            test_csv = p

    return train_csv, test_csv


def choose_fullbody_train_test_csv(csv_files):
    train_csv = None
    test_csv = None

    for p in csv_files:
        name = os.path.basename(p).lower()
        if name == "fullbody_train.csv":
            train_csv = p
        if name == "fullbody_test.csv":
            test_csv = p

    if train_csv is None or test_csv is None:
        train_csv, test_csv = choose_train_test_csv(csv_files)

    return train_csv, test_csv


def parse_bbox_value(v):
    if pd.isna(v):
        return None

    if isinstance(v, (list, tuple, np.ndarray)):
        nums = list(v)
    else:
        s = str(v).strip()

        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                nums = list(parsed)
            else:
                nums = None
        except Exception:
            nums = None

        if nums is None:
            nums = re.findall(r"[-+]?\d*\.\d+|[-+]?\d+", s)

    if nums is None or len(nums) < 4:
        return None

    nums = [float(x) for x in nums[:4]]
    a, b, c, d = nums

    # Normalized YOLO xywh
    if max(abs(a), abs(b), abs(c), abs(d)) <= 1.5:
        x_center, y_center, bw, bh = a, b, c, d
        x1 = x_center - bw / 2
        y1 = y_center - bh / 2
        x2 = x_center + bw / 2
        y2 = y_center + bh / 2
        return (x1, y1, x2, y2)

    # If already xyxy-like
    if c > a and d > b:
        return (a, b, c, d)

    # Fallback xywh
    x, y, w, h = a, b, c, d
    return (x, y, x + w, y + h)


def build_bbox_map_from_csv(csv_path):
    if csv_path is None or not os.path.exists(csv_path):
        return {}

    df = pd.read_csv(csv_path)
    cols = df.columns.tolist()
    lower_cols = {c.lower(): c for c in cols}

    image_col = None
    for c in ["image", "filename", "file", "image_id", "name"]:
        if c in lower_cols:
            image_col = lower_cols[c]
            break

    if image_col is None:
        for c in cols:
            if (
                df[c]
                .astype(str)
                .str.contains(".jpg|.png|.jpeg", case=False, regex=True)
                .any()
            ):
                image_col = c
                break

    if image_col is None:
        print("Cannot find image column in:", csv_path)
        print("Columns:", cols)
        return {}

    bbox_col = None
    for c in ["bbox", "box"]:
        if c in lower_cols:
            bbox_col = lower_cols[c]
            break

    bbox_map = {}

    if bbox_col is not None:
        for _, row in df.iterrows():
            image_name = os.path.basename(str(row[image_col]))
            bbox = parse_bbox_value(row[bbox_col])
            if bbox is not None:
                bbox_map[image_name] = bbox

        print(f"Parsed {len(bbox_map)} bbox rows from {csv_path}")
        print(
            "Example:",
            next(iter(bbox_map.items())) if len(bbox_map) > 0 else None,
        )
        return bbox_map

    # Try separate bbox columns
    candidate_sets = [
        ("x_min", "y_min", "x_max", "y_max"),
        ("xmin", "ymin", "xmax", "ymax"),
        ("x1", "y1", "x2", "y2"),
        ("left", "top", "right", "bottom"),
        ("x", "y", "w", "h"),
        ("x", "y", "width", "height"),
    ]

    for keys in candidate_sets:
        if all(k in lower_cols for k in keys):
            c1, c2, c3, c4 = [lower_cols[k] for k in keys]

            for _, row in df.iterrows():
                image_name = os.path.basename(str(row[image_col]))
                vals = [row[c1], row[c2], row[c3], row[c4]]
                bbox = parse_bbox_value(vals)
                if bbox is not None:
                    bbox_map[image_name] = bbox

            print(f"Parsed {len(bbox_map)} bbox rows from {csv_path}")
            print(
                "Example:",
                next(iter(bbox_map.items())) if len(bbox_map) > 0 else None,
            )
            return bbox_map

    print("Cannot find bbox column in:", csv_path)
    print("Columns:", cols)
    return {}


def build_image_path_map(root_dir):
    path_map = {}
    if root_dir is None:
        return path_map

    for root, dirs, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                path_map[f] = os.path.join(root, f)

    return path_map

In [ ]:
# ============================================================
# Fullbody bbox maps
# ============================================================


train_fullbody_map = {}
test_fullbody_map = {}

if CFG.use_fullbody_crop and FULLBODY_DIR is not None:
    fullbody_csvs = find_csv_files(FULLBODY_DIR)
    print("Fullbody CSVs:", fullbody_csvs)

    fb_train_csv, fb_test_csv = choose_fullbody_train_test_csv(fullbody_csvs)
    print("Chosen fullbody train csv:", fb_train_csv)
    print("Chosen fullbody test csv:", fb_test_csv)

    train_fullbody_map = build_bbox_map_from_csv(fb_train_csv)
    test_fullbody_map = build_bbox_map_from_csv(fb_test_csv)

    print("train_fullbody_map:", len(train_fullbody_map))
    print("test_fullbody_map:", len(test_fullbody_map))

    if len(train_fullbody_map) == 0 or len(test_fullbody_map) == 0:
        CFG.use_fullbody_crop = False

print("CFG.use_fullbody_crop:", CFG.use_fullbody_crop)

In [ ]:
# ============================================================
# Backfin source: cropped image paths or bbox CSV
# ============================================================

train_backfin_path_map = {}
test_backfin_path_map = {}
train_backfin_bbox_map = {}
test_backfin_bbox_map = {}

if CFG.use_backfin_crop and BACKFIN_DIR is not None:
    print("Scanning backfin dataset...")
    backfin_csvs = find_csv_files(BACKFIN_DIR)
    backfin_img_map = build_image_path_map(BACKFIN_DIR)

    print("Backfin CSVs:", backfin_csvs[:10])
    print("Backfin image count:", len(backfin_img_map))

    # If cropped images exist, use them directly.
    if len(backfin_img_map) > 0:
        train_images_set = set(os.listdir(TRAIN_IMG_DIR))
        test_images_set = set(os.listdir(TEST_IMG_DIR))

        for image_name, path in backfin_img_map.items():
            if image_name in train_images_set:
                train_backfin_path_map[image_name] = path
            elif image_name in test_images_set:
                test_backfin_path_map[image_name] = path

        print("train_backfin_path_map:", len(train_backfin_path_map))
        print("test_backfin_path_map:", len(test_backfin_path_map))

    # Also try CSV bbox fallback.
    if len(train_backfin_path_map) == 0 or len(test_backfin_path_map) == 0:
        bf_train_csv, bf_test_csv = choose_train_test_csv(backfin_csvs)
        print("Chosen backfin train csv:", bf_train_csv)
        print("Chosen backfin test csv:", bf_test_csv)

        train_backfin_bbox_map = build_bbox_map_from_csv(bf_train_csv)
        test_backfin_bbox_map = build_bbox_map_from_csv(bf_test_csv)

        print("train_backfin_bbox_map:", len(train_backfin_bbox_map))
        print("test_backfin_bbox_map:", len(test_backfin_bbox_map))

    if (
        len(train_backfin_path_map) == 0
        and len(test_backfin_path_map) == 0
        and len(train_backfin_bbox_map) == 0
        and len(test_backfin_bbox_map) == 0
    ):
        print("Backfin source unavailable. Disable backfin crop.")
        CFG.use_backfin_crop = False

print("CFG.use_backfin_crop:", CFG.use_backfin_crop)

## Build Datasets (Required for Training and Inference)


In [ ]:
# ============================================================
# Load official train/test
# ============================================================

# Load the official train/test metadata.
train_df = pd.read_csv(TRAIN_CSV)
sample_df = pd.read_csv(SAMPLE_SUB_CSV)

train_df["path"] = train_df["image"].apply(
    lambda x: os.path.join(TRAIN_IMG_DIR, x)
)
sample_df["path"] = sample_df["image"].apply(
    lambda x: os.path.join(TEST_IMG_DIR, x)
)


# Fix known species label issues.
train_df["species"] = train_df["species"].replace(
    {
        "globis": "short_finned_pilot_whale",
        "pilot_whale": "short_finned_pilot_whale",
        "kiler_whale": "killer_whale",
        "bottlenose_dolpin": "bottlenose_dolphin",
    }
)

# Individual label encoder
le_id = LabelEncoder()
train_df["label"] = le_id.fit_transform(train_df["individual_id"])
num_classes = train_df["label"].nunique()
np.save(os.path.join(CFG.output_dir, "label_classes.npy"), le_id.classes_)

# Species label encoder
le_species = LabelEncoder()
train_df["label_species"] = le_species.fit_transform(train_df["species"])
num_species_classes = train_df["label_species"].nunique()
np.save(
    os.path.join(CFG.output_dir, "species_classes.npy"), le_species.classes_
)

print("num_classes:", num_classes)
print("num_species_classes:", num_species_classes)

# ============================================================
# Build train_all_df = real train + high-confidence pseudo labels
# ============================================================

keep_cols = [
    "image",
    "path",
    "individual_id",
    "species",
    "label",
    "label_species",
    "is_pseudo",
    "pseudo_score",
]

# Real train rows
train_df["is_pseudo"] = 0
train_df["pseudo_score"] = 1.0

if getattr(CFG, "use_pseudo", False):
    pseudo_csv_candidates = [
        getattr(CFG, "pseudo_csv", None),
        os.path.join(CFG.output_dir, "pseudo.csv"),
        "data/pseudo.csv",
        "pseudo.csv",
        "/kaggle/working/pseudo.csv",
    ]
    pseudo_csv_candidates = [
        str(p) for p in pseudo_csv_candidates if p is not None
    ]
    pseudo_csv_path = next(
        (p for p in pseudo_csv_candidates if os.path.exists(p)), None
    )

    if pseudo_csv_path is None:
        raise FileNotFoundError(
            "CFG.use_pseudo=True, but pseudo.csv was not found. Tried: "
            + str(pseudo_csv_candidates)
            + "\nPlease upload pseudo.csv or set CFG.pseudo_csv / PSEUDO_CSV."
        )

    print("Using pseudo CSV:", pseudo_csv_path)

    pseudo_raw = pd.read_csv(pseudo_csv_path)
    print("Raw pseudo:", pseudo_raw.shape)
    print(pseudo_raw.head())

    required_cols = [
        "image",
        "top1_pred",
        "top1_score",
        "top2_pred",
        "top2_score",
        "score_margin_top1_top2",
    ]
    missing_cols = [c for c in required_cols if c not in pseudo_raw.columns]
    if len(missing_cols) > 0:
        raise ValueError(f"pseudo.csv missing columns: {missing_cols}")

    pseudo_df = pseudo_raw.copy()

    # 1) Do not add new_individual into a closed-set classifier.
    if getattr(CFG, "pseudo_drop_new_individual", True):
        pseudo_df = pseudo_df[
            pseudo_df["top1_pred"] != "new_individual"
        ].copy()

    # 2) Only keep IDs that already exist in official train labels.
    known_ids = set(train_df["individual_id"].unique())
    pseudo_df = pseudo_df[pseudo_df["top1_pred"].isin(known_ids)].copy()

    # 3) High-confidence filter: top1 score high enough and top1-top2 gap large enough.
    pseudo_df = pseudo_df[
        (pseudo_df["top1_score"] >= CFG.pseudo_top1_threshold)
        & (pseudo_df["score_margin_top1_top2"] >= CFG.pseudo_margin_threshold)
    ].copy()

    print("Filtered pseudo before per-ID limit:", pseudo_df.shape)

    # 4) top1_pred becomes the pseudo individual_id.
    pseudo_df["individual_id"] = pseudo_df["top1_pred"]
    pseudo_df["pseudo_score"] = pseudo_df["top1_score"]
    pseudo_df["is_pseudo"] = 1

    # 5) Per-ID cap: avoid over-strengthening a few frequent IDs.
    max_per_id = getattr(CFG, "pseudo_max_per_id", None)
    if max_per_id is not None and max_per_id > 0:
        pseudo_df = (
            pseudo_df.sort_values(
                ["individual_id", "top1_score", "score_margin_top1_top2"],
                ascending=[True, False, False],
            )
            .groupby("individual_id")
            .head(max_per_id)
            .reset_index(drop=True)
        )

    print("Filtered pseudo after per-ID limit:", pseudo_df.shape)
    print("Pseudo unique individual_id:", pseudo_df["individual_id"].nunique())

    id_counts = pseudo_df["individual_id"].value_counts()
    print("\nPseudo count statistics per ID:")
    print(id_counts.describe())
    print("\nTop 20 pseudo-heavy IDs:")
    print(id_counts.head(20))

    # 6) Pseudo images come from test_images.
    pseudo_df["path"] = pseudo_df["image"].apply(
        lambda x: os.path.join(TEST_IMG_DIR, x)
    )

    # Safety check: drop pseudo rows whose image file is not available.
    before = len(pseudo_df)
    pseudo_df = pseudo_df[pseudo_df["path"].apply(os.path.exists)].copy()
    print("Drop pseudo without image file:", before - len(pseudo_df))

    # 7) Map individual_id back to species using official train labels.
    id_to_species = (
        train_df.groupby("individual_id")["species"]
        .agg(lambda x: x.mode().iloc[0])
        .to_dict()
    )
    pseudo_df["species"] = pseudo_df["individual_id"].map(id_to_species)

    before = len(pseudo_df)
    pseudo_df = pseudo_df.dropna(subset=["species"]).copy()
    print("Drop pseudo without species:", before - len(pseudo_df))

    # 8) Use the SAME LabelEncoders fitted on official train.
    pseudo_df["label"] = le_id.transform(pseudo_df["individual_id"])
    pseudo_df["label_species"] = le_species.transform(pseudo_df["species"])

    pseudo_train_df = pseudo_df[keep_cols].copy()
    train_real_df = train_df[keep_cols].copy()

    train_all_df = pd.concat(
        [train_real_df, pseudo_train_df],
        axis=0,
        ignore_index=True,
    )
else:
    train_all_df = train_df[
        keep_cols
    ].copy()  # Keep the training data the same as the original set.

# ============================================================
# Count images per class for adaptive margin calculation.
# ============================================================

id_class_nums = (
    train_all_df["label"]
    .value_counts()
    .reindex(range(num_classes), fill_value=0)
    .sort_index()
    .values
)

species_class_nums = (
    train_all_df["label_species"]
    .value_counts()
    .reindex(range(num_species_classes), fill_value=0)
    .sort_index()
    .values
)

# ============================================================
# Summary check
# ============================================================

print("\n" + "=" * 60)
print("Class count arrays")
print("=" * 60)
print("id_class_nums:", id_class_nums.shape)
print("species_class_nums:", species_class_nums.shape)

print("\n" + "=" * 60)
print("Dataset size")
print("=" * 60)
print("Original train:", train_df.shape)
print("Train all:", train_all_df.shape)
print("Pseudo count:", int(train_all_df["is_pseudo"].sum()))
print("Test:", sample_df.shape)

print("\n" + "=" * 60)
print("Number of classes")
print("=" * 60)
print("num_classes:", num_classes)
print("num_species_classes:", num_species_classes)

print("\n" + "=" * 60)
print("Crop settings")
print("=" * 60)
print("crop_probs:", CFG.crop_probs)

In [ ]:
# ============================================================
# Transforms
# ============================================================


# Training transforms.
def get_train_transforms():
    return A.Compose(
        [
            A.Resize(CFG.image_size, CFG.image_size),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.05,
                scale_limit=0.15,
                rotate_limit=10,
                border_mode=cv2.BORDER_CONSTANT,
                p=0.5,
            ),
            A.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.2,
                hue=0.05,
                p=0.5,
            ),
            A.CoarseDropout(
                max_holes=8,
                max_height=CFG.image_size // 12,
                max_width=CFG.image_size // 12,
                p=0.3,
            ),
            A.Normalize(),
            ToTensorV2(),
        ]
    )


# Validation and inference transforms.


def get_valid_transforms():
    return A.Compose(
        [
            A.Resize(CFG.image_size, CFG.image_size),
            A.Normalize(),
            ToTensorV2(),
        ]
    )

In [ ]:
# ============================================================
# Dataset
# ============================================================


def crop_by_bbox(image, bbox, margin=0.05):
    if bbox is None:
        return image

    h, w = image.shape[:2]
    x1, y1, x2, y2 = bbox

    if max(abs(x1), abs(y1), abs(x2), abs(y2)) <= 2.0:
        x1 *= w
        x2 *= w
        y1 *= h
        y2 *= h

    bw = x2 - x1
    bh = y2 - y1

    x1 = int(max(0, x1 - bw * margin))
    y1 = int(max(0, y1 - bh * margin))
    x2 = int(min(w, x2 + bw * margin))
    y2 = int(min(h, y2 + bh * margin))

    if x2 <= x1 or y2 <= y1:
        return image

    cropped = image[y1:y2, x1:x2]
    if cropped.size == 0:
        return image

    return cropped


class HappyWhaleDataset(Dataset):
    def __init__(
        self, df, transforms=None, is_test=False, mode="train_random"
    ):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms
        self.is_test = is_test
        self.mode = mode

        self.crop_names = list(CFG.crop_probs.keys())
        self.crop_probs = np.array(
            list(CFG.crop_probs.values()), dtype=np.float64
        )
        self.crop_probs = self.crop_probs / self.crop_probs.sum()

    def __len__(self):
        return len(self.df)

    def choose_crop_mode(self):
        if self.mode == "train_random":
            return np.random.choice(self.crop_names, p=self.crop_probs)
        return self.mode

    def read_original_image(self, image_path):
        image = cv2.imread(image_path)
        if image is None:
            raise FileNotFoundError(f"Image not found: {image_path}")
        return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    def read_backfin_image_if_available(self, image_name):
        if self.is_test:
            p = test_backfin_path_map.get(image_name, None)
        else:
            # Real train images are in train_backfin_path_map.
            p = train_backfin_path_map.get(image_name, None)

            # Pseudo images come from test_images, so fallback to test_backfin_path_map.
            if p is None:
                p = test_backfin_path_map.get(image_name, None)

        if p is None:
            return None

        image = cv2.imread(p)
        if image is None:
            return None

        return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    def get_bbox(self, image_name, crop_mode):
        if crop_mode == "fullbody":
            if self.is_test:
                return test_fullbody_map.get(image_name, None)

            # Real train images are in train_fullbody_map.
            bbox = train_fullbody_map.get(image_name, None)

            # Pseudo images come from test_images, so fallback to test_fullbody_map.
            if bbox is None:
                bbox = test_fullbody_map.get(image_name, None)

            return bbox

        if crop_mode == "backfin":
            if self.is_test:
                return test_backfin_bbox_map.get(image_name, None)

            # Real train images are in train_backfin_bbox_map.
            bbox = train_backfin_bbox_map.get(image_name, None)

            # Pseudo images come from test_images, so fallback to test_backfin_bbox_map.
            if bbox is None:
                bbox = test_backfin_bbox_map.get(image_name, None)

            return bbox

        return None

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_name = row["image"]
        image_path = row["path"]

        crop_mode = self.choose_crop_mode()

        image = None

        if crop_mode == "backfin" and CFG.use_backfin_crop:
            image = self.read_backfin_image_if_available(image_name)

        if image is None:
            image = self.read_original_image(image_path)

            if crop_mode == "fullbody" and CFG.use_fullbody_crop:
                bbox = self.get_bbox(image_name, "fullbody")
                image = crop_by_bbox(image, bbox, margin=CFG.crop_margin)

            elif crop_mode == "backfin" and CFG.use_backfin_crop:
                bbox = self.get_bbox(image_name, "backfin")
                image = crop_by_bbox(image, bbox, margin=CFG.crop_margin)

        if self.transforms:
            image = self.transforms(image=image)["image"]

        if self.is_test:
            return image, image_name

        label = int(row["label"])
        label_species = int(row["label_species"])
        return image, label, label_species


# Debug crop check
print("Debug crop check:")
for mode in ["none", "fullbody", "backfin"]:
    ds = HappyWhaleDataset(
        train_all_df.head(5), transforms=None, is_test=False, mode=mode
    )
    row = train_all_df.iloc[0]
    image, label, label_species = ds[0]
    print(
        mode,
        row["image"],
        image.shape,
        "label:",
        label,
        "species:",
        label_species,
    )

## Build Training Datasets and Model

In [ ]:
# ============================================================
# DataLoaders
# ============================================================

train_loader = DataLoader(
    HappyWhaleDataset(
        train_all_df,
        transforms=get_train_transforms(),
        is_test=False,
        mode="train_random",
    ),
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=True,
    drop_last=True,
)

In [ ]:
# ============================================================
# Model
# ============================================================


class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6, requires_grad=True):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p, requires_grad=requires_grad)
        self.eps = eps

    def forward(self, x):
        return (
            x.clamp(min=self.eps).pow(self.p).mean((-2, -1)).pow(1.0 / self.p)
        )


class ArcMarginProductSubcenter(nn.Module):
    def __init__(self, in_features, out_features, k=3):
        super().__init__()
        self.weight = nn.Parameter(
            torch.FloatTensor(out_features * k, in_features)
        )
        self.k = k
        self.out_features = out_features
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1.0 / math.sqrt(self.weight.size(1))
        self.weight.data.uniform_(-stdv, stdv)

    def forward(self, features):
        cosine_all = F.linear(F.normalize(features), F.normalize(self.weight))
        cosine_all = cosine_all.view(-1, self.out_features, self.k)
        cosine, _ = torch.max(cosine_all, dim=2)
        return cosine


class ArcFaceLossAdaptiveMargin(nn.Module):
    def __init__(self, margins, n_classes, s=30.0):
        super().__init__()
        self.s = s
        self.out_dim = n_classes
        self.register_buffer(
            "margins", torch.tensor(margins, dtype=torch.float32)
        )

    def forward(self, logits, labels):
        ms = self.margins[labels]
        cos_m = torch.cos(ms)
        sin_m = torch.sin(ms)
        th = torch.cos(math.pi - ms)
        mm = torch.sin(math.pi - ms) * ms

        labels_onehot = F.one_hot(labels, self.out_dim).float()

        logits = logits.float()
        cosine = logits.clamp(-1 + 1e-7, 1 - 1e-7)
        sine = torch.sqrt(torch.clamp(1.0 - torch.pow(cosine, 2), min=1e-7))

        phi = cosine * cos_m.view(-1, 1) - sine * sin_m.view(-1, 1)
        phi = torch.where(
            cosine > th.view(-1, 1), phi, cosine - mm.view(-1, 1)
        )

        return (
            (labels_onehot * phi) + ((1.0 - labels_onehot) * cosine)
        ) * self.s


def make_adaptive_margins(class_nums, power, coef, cons):
    class_nums = np.asarray(class_nums, dtype=np.float32)
    margins = np.power(class_nums, power) * coef + cons
    return margins.astype(np.float32)


class HappyWhaleModel(nn.Module):
    def __init__(
        self,
        model_name,
        num_classes,
        num_species_classes,
        id_class_nums,
        species_class_nums,
    ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,
            global_pool="",
        )

        in_features = self.backbone.num_features

        self.pool = GeM(p=3.0, requires_grad=True)

        self.embedding = nn.Sequential(
            nn.BatchNorm1d(in_features),
        )

        self.head_id = ArcMarginProductSubcenter(
            in_features,
            num_classes,
            k=CFG.n_center_id,
        )

        self.head_species = ArcMarginProductSubcenter(
            in_features,
            num_species_classes,
            k=CFG.n_center_species,
        )

        margins_id = make_adaptive_margins(
            id_class_nums,
            CFG.margin_power_id,
            CFG.margin_coef_id,
            CFG.margin_cons_id,
        )

        margins_species = make_adaptive_margins(
            species_class_nums,
            CFG.margin_power_species,
            CFG.margin_coef_species,
            CFG.margin_cons_species,
        )

        print(
            "margins_id:",
            margins_id[:10],
            "min:",
            margins_id.min(),
            "max:",
            margins_id.max(),
        )
        print(
            "margins_species:",
            margins_species,
            "min:",
            margins_species.min(),
            "max:",
            margins_species.max(),
        )

        self.margin_fn_id = ArcFaceLossAdaptiveMargin(
            margins_id,
            num_classes,
            s=CFG.s_id,
        )

        self.margin_fn_species = ArcFaceLossAdaptiveMargin(
            margins_species,
            num_species_classes,
            s=CFG.s_species,
        )

    def get_feat(self, x):
        feat = self.backbone(x)
        feat = self.pool(feat)
        feat = self.embedding(feat)
        feat = F.normalize(feat)
        return feat

    def forward(self, x):
        feat = self.get_feat(x)
        logits_id = self.head_id(feat)
        logits_species = self.head_species(feat)
        return logits_id, logits_species, feat

In [ ]:
# ============================================================
# Loss: Cross Entropy / Focal Loss
# ============================================================


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, logits, targets):
        """
        logits:  [batch_size, num_classes]
        targets: [batch_size]
        """
        ce_loss = F.cross_entropy(
            logits,
            targets,
            weight=self.alpha,
            reduction="none",
        )

        pt = torch.exp(-ce_loss)
        focal_loss = ((1.0 - pt) ** self.gamma) * ce_loss

        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        else:
            return focal_loss


def make_class_balanced_alpha(class_counts, beta=0.999, device="cuda"):
    counts = np.asarray(class_counts, dtype=np.float32)
    counts = np.maximum(counts, 1.0)

    effective_num = 1.0 - np.power(beta, counts)
    alpha = (1.0 - beta) / effective_num
    alpha = alpha / alpha.mean()

    alpha = torch.tensor(alpha, dtype=torch.float32, device=device)
    return alpha

In [ ]:
# ============================================================
# Build model
# ============================================================


model = HappyWhaleModel(
    CFG.model_name,
    num_classes,
    num_species_classes,
    id_class_nums,
    species_class_nums,
)

model = model.to(CFG.device)

optimizer = torch.optim.AdamW(
    [
        {"params": model.backbone.parameters(), "lr": CFG.lr_backbone},
        {"params": model.pool.parameters(), "lr": CFG.lr_backbone},
        {"params": model.embedding.parameters(), "lr": CFG.lr_head},
        {"params": model.head_id.parameters(), "lr": CFG.lr_head},
        {"params": model.head_species.parameters(), "lr": CFG.lr_head},
    ],
    weight_decay=CFG.weight_decay,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG.epochs,
    eta_min=1e-6,
)

# ============================================================
# Criterion
# ============================================================

if CFG.use_focal_loss:
    if CFG.use_focal_alpha:
        alpha_id = make_class_balanced_alpha(
            id_class_nums,
            beta=CFG.focal_alpha_beta_id,
            device=CFG.device,
        )

        alpha_species = make_class_balanced_alpha(
            species_class_nums,
            beta=CFG.focal_alpha_beta_species,
            device=CFG.device,
        )
    else:
        alpha_id = None
        alpha_species = None

    print("alpha_id:", None if alpha_id is None else alpha_id.shape)
    print(
        "alpha_species:",
        None if alpha_species is None else alpha_species.shape,
    )

    if alpha_id is not None:
        assert alpha_id.shape[0] == num_classes

    if alpha_species is not None:
        assert alpha_species.shape[0] == num_species_classes

    criterion_id = FocalLoss(
        gamma=CFG.focal_gamma_id,
        alpha=alpha_id,
        reduction="mean",
    )

    criterion_species = FocalLoss(
        gamma=CFG.focal_gamma_species,
        alpha=alpha_species,
        reduction="mean",
    )

    print("Using Focal Loss")
    print("focal_gamma_id:", CFG.focal_gamma_id)
    print("focal_gamma_species:", CFG.focal_gamma_species)
    print("use_focal_alpha:", CFG.use_focal_alpha)

else:
    criterion_id = nn.CrossEntropyLoss()
    criterion_species = nn.CrossEntropyLoss()

    print("Using CrossEntropyLoss")

scaler = torch.cuda.amp.GradScaler(enabled=(CFG.device == "cuda"))


# ============================================================
# Checkpoint / log paths
# ============================================================

last_ckpt_path = os.path.join(CFG.output_dir, "last_checkpoint.pth")
training_log_path = os.path.join(CFG.output_dir, "training_log.csv")

checkpoint_dir = os.path.join(CFG.output_dir, "checkpoints")
figure_dir = os.path.join(CFG.output_dir, "figures")

os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(figure_dir, exist_ok=True)

In [ ]:
# ============================================================
# Resume from local or Hugging Face
# ============================================================

try:
    from huggingface_hub import hf_hub_download, login
except ImportError:
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"]
    )
    from huggingface_hub import hf_hub_download, login


# ------------------------------------------------------------
# Hugging Face settings
# ------------------------------------------------------------

CFG.resume = True
CFG.resume_from_hf = True
CFG.hf_checkpoint_file = "checkpoints/last_checkpoint.pth"  # Change this to choose which checkpoint to download.


def hf_login_if_needed():
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")

    if not token:
        try:
            from google.colab import userdata

            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None

    if token:
        login(token=token)
        return token

    # If HF_TOKEN is not configured in Colab userdata, paste the token manually.
    from getpass import getpass

    token = getpass("Paste your Hugging Face access token: ")
    login(token=token)
    return token


def download_checkpoint_from_hf_if_needed(local_ckpt_path):
    local_ckpt_path = Path(local_ckpt_path)
    local_ckpt_path.parent.mkdir(parents=True, exist_ok=True)

    if getattr(CFG, "resume_from_hf", True):
        print("Local checkpoint not found. Download from Hugging Face...")
        token = (
            hf_login_if_needed() if getattr(CFG, "hf_private", True) else None
        )

        downloaded_path = hf_hub_download(
            repo_id=CFG.hf_repo_id,
            repo_type=CFG.hf_repo_type,
            filename=CFG.hf_checkpoint_file,
            token=token,
        )

        # Do not modify files inside the HF cache directly; copy the checkpoint to the local checkpoint directory.
        shutil.copy(downloaded_path, local_ckpt_path)

        print("Downloaded HF checkpoint:", CFG.hf_checkpoint_file)
        print("Saved to local:", local_ckpt_path)

        return local_ckpt_path

    # Use the local checkpoint if it already exists.
    if local_ckpt_path.exists():
        print("Use local checkpoint:", local_ckpt_path)
        return local_ckpt_path

    print("No local checkpoint and resume_from_hf=False.")
    return local_ckpt_path


# ============================================================
# Resume
# ============================================================


start_epoch = 0
history = []

# Try downloading the checkpoint from Hugging Face to last_ckpt_path first.
if CFG.resume:
    last_ckpt_path = download_checkpoint_from_hf_if_needed(last_ckpt_path)

if CFG.resume and os.path.exists(last_ckpt_path):
    print("Loading checkpoint:", last_ckpt_path)
    checkpoint = torch.load(last_ckpt_path, map_location=CFG.device)
    compatible = (
        checkpoint.get("model_name") == CFG.model_name
        and checkpoint.get("image_size") == CFG.image_size
        and checkpoint.get("num_classes") == num_classes
    )

    if compatible:
        model.load_state_dict(checkpoint["model"])
        print("Loaded model weights.")

        if not CFG.use_pseudo:
            if (
                "optimizer" in checkpoint
                and checkpoint["optimizer"] is not None
            ):
                optimizer.load_state_dict(checkpoint["optimizer"])
                print("Loaded optimizer state.")
            else:
                print("No optimizer state in checkpoint.")

            if (
                "scheduler" in checkpoint
                and checkpoint["scheduler"] is not None
            ):
                scheduler.load_state_dict(checkpoint["scheduler"])
                print("Loaded scheduler state.")
            else:
                print("No scheduler state in checkpoint.")

            if "scaler" in checkpoint and checkpoint["scaler"] is not None:
                scaler.load_state_dict(checkpoint["scaler"])
                print("Loaded AMP scaler state.")

            else:
                print("No scaler state in checkpoint.")

            start_epoch = int(checkpoint["epoch"]) + 1
            history = checkpoint.get("history", [])

            print("Resume mode: continue")
            print("Resume from epoch:", start_epoch)

        elif CFG.use_pseudo:
            # Stage 2: load only the model; reinitialize optimizer, scheduler, and scaler.
            start_epoch = 0
            history = []

            print("Resume mode: stage2")
            print("Loaded model weights only.")
            print("Original checkpoint epoch:", checkpoint.get("epoch"))
            print("Stage-2 start_epoch:", start_epoch + 1)
            print("Optimizer / scheduler / scaler are newly initialized.")
            print("Current lr_backbone:", CFG.lr_backbone)
            print("Current lr_head:", CFG.lr_head)

    else:
        print("Checkpoint not compatible. Start from scratch.")
        print("Checkpoint config:")
        print("model_name:", checkpoint.get("model_name"))
        print("image_size:", checkpoint.get("image_size"))
        print("num_classes:", checkpoint.get("num_classes"))
        print("num_species_classes:", checkpoint.get("num_species_classes"))
        print("crop_probs:", checkpoint.get("crop_probs"))

        print("Current config:")
        print("model_name:", CFG.model_name)
        print("image_size:", CFG.image_size)
        print("num_classes:", num_classes)
        print("num_species_classes:", num_species_classes)
        print("crop_probs:", CFG.crop_probs)

else:
    print("No checkpoint found. Start from scratch.")

In [ ]:
# ============================================================
# Training
# ============================================================


def plot_training_log(log_path=None, save_dir=None):
    import os

    import matplotlib.pyplot as plt
    import pandas as pd

    log_path = log_path or training_log_path
    save_dir = save_dir or figure_dir
    os.makedirs(save_dir, exist_ok=True)

    if not os.path.exists(log_path):
        print("Training log not found:", log_path)
        return None, None

    log_df = pd.read_csv(log_path)

    # Loss curve
    loss_fig_path = os.path.join(save_dir, "training_loss_curve.png")

    plt.figure(figsize=(8, 5))
    plt.plot(log_df["epoch"], log_df["train_loss"], label="total loss")
    plt.plot(log_df["epoch"], log_df["train_loss_id"], label="id loss")
    plt.plot(
        log_df["epoch"], log_df["train_loss_species"], label="species loss"
    )
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Loss Curve")
    plt.legend()
    plt.grid(True)
    plt.savefig(loss_fig_path, dpi=200, bbox_inches="tight")
    plt.close()

    # LR curve
    lr_fig_path = os.path.join(save_dir, "learning_rate_curve.png")

    plt.figure(figsize=(8, 5))
    plt.plot(log_df["epoch"], log_df["lr_backbone"], label="backbone lr")
    plt.plot(log_df["epoch"], log_df["lr_head"], label="head lr")
    plt.xlabel("Epoch")
    plt.ylabel("Learning Rate")
    plt.title("Learning Rate Schedule")
    plt.legend()
    plt.grid(True)
    plt.savefig(lr_fig_path, dpi=200, bbox_inches="tight")
    plt.close()

    print("Saved figures:")
    print(loss_fig_path)
    print(lr_fig_path)

    return loss_fig_path, lr_fig_path


def build_checkpoint(epoch, history):
    return {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        # config / metadata
        "model_name": CFG.model_name,
        "image_size": CFG.image_size,
        "num_classes": num_classes,
        "num_species_classes": num_species_classes,
        "crop_probs": CFG.crop_probs,
        "loss_id_ratio": CFG.loss_id_ratio,
        # focal loss config
        "use_focal_loss": getattr(CFG, "use_focal_loss", False),
        "focal_gamma_id": getattr(CFG, "focal_gamma_id", None),
        "focal_gamma_species": getattr(CFG, "focal_gamma_species", None),
        "use_focal_alpha": getattr(CFG, "use_focal_alpha", False),
        "focal_alpha_beta_id": getattr(CFG, "focal_alpha_beta_id", None),
        "focal_alpha_beta_species": getattr(
            CFG, "focal_alpha_beta_species", None
        ),
        # data info
        "train_size": len(train_all_df),
        "use_pseudo": getattr(CFG, "use_pseudo", False),
        "pseudo_top1_threshold": getattr(CFG, "pseudo_top1_threshold", None),
        "pseudo_margin_threshold": getattr(
            CFG, "pseudo_margin_threshold", None
        ),
        "pseudo_max_per_id": getattr(CFG, "pseudo_max_per_id", None),
        "pseudo_count": int(train_all_df["is_pseudo"].sum()),
        # training history
        "history": history,
    }


def train_one_epoch(epoch):
    model.train()

    total_loss = 0.0
    total_loss_id = 0.0
    total_loss_species = 0.0

    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{CFG.epochs} [train]")

    for step, (images, labels, labels_species) in enumerate(pbar):
        images = images.to(CFG.device, non_blocking=True)
        labels = labels.to(CFG.device, non_blocking=True)
        labels_species = labels_species.to(CFG.device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=(CFG.device == "cuda")):
            logits_id, logits_species, feat = model(images)

            margin_logits_id = model.margin_fn_id(logits_id, labels)
            margin_logits_species = model.margin_fn_species(
                logits_species, labels_species
            )

            loss_id = criterion_id(margin_logits_id, labels)
            loss_species = criterion_species(
                margin_logits_species, labels_species
            )

            loss = (
                CFG.loss_id_ratio * loss_id
                + (1.0 - CFG.loss_id_ratio) * loss_species
            )

            # Important: divide the loss by grad_accum_steps for gradient accumulation.
            loss_for_backward = loss / CFG.grad_accum_steps

        scaler.scale(loss_for_backward).backward()

        # Update parameters every grad_accum_steps batches.
        do_step = (step + 1) % CFG.grad_accum_steps == 0

        # Also update on the final batch even if it does not reach grad_accum_steps.
        is_last_step = (step + 1) == len(train_loader)

        if do_step or is_last_step:
            scaler.unscale_(optimizer)

            if getattr(CFG, "grad_clip", None) is not None:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), CFG.grad_clip
                )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        # Log the original loss, not the scaled loss_for_backward.
        total_loss += loss.item()
        total_loss_id += loss_id.item()
        total_loss_species += loss_species.item()

        avg_loss = total_loss / (step + 1)
        avg_id = total_loss_id / (step + 1)
        avg_sp = total_loss_species / (step + 1)

        pbar.set_postfix(
            loss=avg_loss,
            id=avg_id,
            sp=avg_sp,
            lr=optimizer.param_groups[0]["lr"],
        )

    return (
        total_loss / len(train_loader),
        total_loss_id / len(train_loader),
        total_loss_species / len(train_loader),
    )

In [ ]:
# ============================================================
# Training
# ============================================================


if CFG.do_train:
    print("Start training...")
    print("Start epoch:", start_epoch + 1)

    # Upload this file because inference depends on it.
    for aux_name in [
        "label_classes.npy",
        "species_classes.npy",
    ]:
        aux_path = os.path.join(CFG.output_dir, aux_name)
        hf_upload_file_if_exists(
            aux_path,
            repo_path=f"metadata/{aux_name}",
            commit_message=f"Upload {aux_name}",
        )

    start_time = time.time()

    for epoch in range(start_epoch, CFG.epochs):
        # Record LR used in this epoch
        lr_backbone_now = optimizer.param_groups[0]["lr"]
        lr_head_now = optimizer.param_groups[2]["lr"]

        train_loss, train_loss_id, train_loss_species = train_one_epoch(epoch)
        scheduler.step()

        elapsed = (time.time() - start_time) / 3600

        row = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_loss_id": train_loss_id,
            "train_loss_species": train_loss_species,
            "elapsed_hours": elapsed,
            "lr_backbone": lr_backbone_now,
            "lr_head": lr_head_now,
            "train_size": len(train_all_df),
            "pseudo_count": int(train_all_df["is_pseudo"].sum()),
            "crop_probs": str(CFG.crop_probs),
        }

        history.append(row)

        print(
            f"Epoch {epoch + 1}: "
            f"loss={train_loss:.6f}, "
            f"id={train_loss_id:.6f}, "
            f"species={train_loss_species:.6f}, "
            f"elapsed={elapsed:.2f}h, "
            f"lr_backbone={lr_backbone_now:.6f}, "
            f"lr_head={lr_head_now:.6f}"
        )

        # Save last checkpoint every epoch
        ckpt = build_checkpoint(epoch=epoch, history=history)
        torch.save(ckpt, last_ckpt_path)

        # Save training log every epoch
        pd.DataFrame(history).to_csv(training_log_path, index=False)

        print("Saved last checkpoint:", last_ckpt_path)
        print("Saved training log:", training_log_path)

        # Plot figures every epoch
        loss_fig_path, lr_fig_path = plot_training_log(
            log_path=training_log_path,
            save_dir=figure_dir,
        )

        # Upload every epoch: latest checkpoint + log + figures
        if CFG.hf_upload_each_epoch:
            hf_upload_file_if_exists(
                last_ckpt_path,
                repo_path="checkpoints/last_checkpoint.pth",
                commit_message=f"Upload latest checkpoint after epoch {epoch + 1}",
            )

            hf_upload_file_if_exists(
                training_log_path,
                repo_path="logs/training_log.csv",
                commit_message=f"Upload training log after epoch {epoch + 1}",
            )

            if loss_fig_path is not None:
                hf_upload_file_if_exists(
                    loss_fig_path,
                    repo_path="logs/figures/training_loss_curve.png",
                    commit_message=f"Upload loss curve after epoch {epoch + 1}",
                )

            if lr_fig_path is not None:
                hf_upload_file_if_exists(
                    lr_fig_path,
                    repo_path="logs/figures/learning_rate_curve.png",
                    commit_message=f"Upload LR curve after epoch {epoch + 1}",
                )

        # Save full checkpoint every 5 epochs
        if (epoch + 1) % 5 == 0:
            epoch_ckpt_name = f"epoch_{epoch + 1:03d}_checkpoint.pth"
            epoch_ckpt_path = os.path.join(checkpoint_dir, epoch_ckpt_name)

            torch.save(ckpt, epoch_ckpt_path)
            print("Saved 5-epoch checkpoint:", epoch_ckpt_path)
            if CFG.hf_upload_each_five_epoch:
                hf_upload_file_if_exists(
                    epoch_ckpt_path,
                    repo_path=f"checkpoints/{epoch_ckpt_name}",
                    commit_message=f"Upload epoch {epoch + 1} checkpoint",
                )

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("Training finished.")
    print("Final latest checkpoint:", last_ckpt_path)
    print("Training log:", training_log_path)
    print("Figures:", figure_dir)
else:
    print("Skip training because CFG.do_train = False")

# Run Embedding Extraction

In [ ]:
# ============================================================
# Inference for each crop mode
# ============================================================


@torch.no_grad()
def extract_champion_outputs_for_mode(crop_mode):
    print(f"\n===== Champion-style inference crop mode: {crop_mode} =====")

    train_loader_inf = DataLoader(
        HappyWhaleDataset(
            train_df,
            transforms=get_valid_transforms(),
            is_test=False,
            mode=crop_mode,
        ),
        batch_size=CFG.infer_batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    test_loader_inf = DataLoader(
        HappyWhaleDataset(
            sample_df,
            transforms=get_valid_transforms(),
            is_test=True,
            mode=crop_mode,
        ),
        batch_size=CFG.infer_batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    def extract(loader, is_test=False):
        model.eval()

        original_indices = []
        labels = []
        labels_species = []
        images_list = []

        feat1_list = []
        feat2_list = []
        pred_logit_list = []
        pred_idx_list = []
        pred_species_logit_list = []
        pred_species_idx_list = []

        offset = 0

        for batch in tqdm(loader, desc=f"Extract {crop_mode}"):
            if is_test:
                images, names = batch
                bs = images.size(0)
                original_indices.extend(list(range(offset, offset + bs)))
                offset += bs
            else:
                images, y, y_species = batch
                bs = images.size(0)
                original_indices.extend(list(range(offset, offset + bs)))
                labels.extend(y.numpy().tolist())
                labels_species.extend(y_species.numpy().tolist())
                offset += bs

            images = images.to(CFG.device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=(CFG.device == "cuda")):
                logits1, species1, feat1 = model(images)
                logits2, species2, feat2 = model(images.flip(3))

                avg_logits = (logits1.float() + logits2.float()) / 2.0
                avg_species = (species1.float() + species2.float()) / 2.0

            topv, topi = torch.topk(avg_logits, k=CFG.logits_topk, dim=1)
            topv_species, topi_species = torch.topk(
                avg_species,
                k=min(CFG.species_topk, avg_species.shape[1]),
                dim=1,
            )

            feat1_list.append(feat1.detach().float().cpu().numpy())
            feat2_list.append(feat2.detach().float().cpu().numpy())
            pred_logit_list.append(
                topv.detach().cpu().numpy().astype("float32")
            )
            pred_idx_list.append(topi.detach().cpu().numpy().astype("int32"))
            pred_species_logit_list.append(
                topv_species.detach().cpu().numpy().astype("float32")
            )
            pred_species_idx_list.append(
                topi_species.detach().cpu().numpy().astype("int32")
            )

            if is_test:
                images_list.extend(list(names))

        ret = {
            "original_index": np.array(
                original_indices
            ),  # Original order index.
            "embed_features1": np.concatenate(
                feat1_list, axis=0
            ),  # Original image embedding.
            "embed_features2": np.concatenate(
                feat2_list, axis=0
            ),  # Flipped image embedding.
            "pred_logit": np.concatenate(
                pred_logit_list, axis=0
            ),  # Individual-head top-k scores.
            "pred_idx": np.concatenate(
                pred_idx_list, axis=0
            ),  # individual head top-k class index
            "pred_species_logit": np.concatenate(
                pred_species_logit_list, axis=0
            ),  # Species-head top-k scores.
            "pred_species_idx": np.concatenate(
                pred_species_idx_list, axis=0
            ),  # species head
        }

        if is_test:
            ret["image"] = np.array(images_list)
        else:
            ret["label"] = np.array(labels)
            ret["label_species"] = np.array(labels_species)

        return ret

    train_results = extract(train_loader_inf, is_test=False)
    test_results = extract(test_loader_inf, is_test=True)

    np.savez_compressed(
        os.path.join(CFG.output_dir, f"train_{crop_mode}_results.npz"),
        **train_results,
    )

    np.savez_compressed(
        os.path.join(CFG.output_dir, f"test_{crop_mode}_results.npz"),
        **test_results,
    )

    train_npz_path = os.path.join(
        CFG.output_dir, f"train_{crop_mode}_results.npz"
    )
    test_npz_path = os.path.join(
        CFG.output_dir, f"test_{crop_mode}_results.npz"
    )
    print("Saved:", train_npz_path)
    print("Saved:", test_npz_path)

    hf_upload_file_if_exists(
        train_npz_path,
        repo_path=f"{CFG.hf_embedding_dir}/train_{crop_mode}_results.npz",
        commit_message=f"Upload train {crop_mode} embeddings",
    )
    hf_upload_file_if_exists(
        test_npz_path,
        repo_path=f"{CFG.hf_embedding_dir}/test_{crop_mode}_results.npz",
        commit_message=f"Upload test {crop_mode} embeddings",
    )

    return train_results, test_results

In [ ]:
# Do embedding


CFG.do_embedding = True
if CFG.do_embedding:
    print("\n========== RUN: DO EMBEDDING ==========")

    for crop_mode in CFG.crop_infer_modes:
        extract_champion_outputs_for_mode(crop_mode)

else:
    print("\n========== SKIP: DO EMBEDDING ==========")

# New Inference Pipeline

In [ ]:
# ============================================================
# Inference utilities
# ============================================================


def normalize_np(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True).clip(min=1e-12)


def restore_all_pred(n_class, pred, pred_idx):
    """
    Restore top-k logits into a full [N, n_class] score matrix.
    """
    n_data = pred.shape[0]
    all_pred = np.zeros((n_data, n_class), dtype=np.float32)

    for i in range(n_data):
        all_pred[i, pred_idx[i]] = pred[i]

    return all_pred

In [ ]:
# ============================================================
# KNN score matrix
# ============================================================


def knn_all_pred_torch(
    n_class,
    test_feat,
    train_feat,
    train_label,
    n_neighbors=500,
    chunk_size=128,
):
    """
    Compute cosine similarity between test features and train features.
    Select the top-k train images for each test image.
    Aggregate image-level similarity into a class-level score matrix.

    Returns:
        out: [N_test, n_class]
    """
    train_feat = normalize_np(train_feat.astype("float32"))
    test_feat = normalize_np(test_feat.astype("float32"))

    train_tensor = torch.tensor(train_feat, dtype=torch.float32).to(CFG.device)
    test_tensor = torch.tensor(test_feat, dtype=torch.float32).to(CFG.device)

    out = np.zeros((len(test_feat), n_class), dtype=np.float32)

    with torch.no_grad():
        for start in tqdm(
            range(0, len(test_tensor), chunk_size),
            desc="KNN all pred",
        ):
            end = min(start + chunk_size, len(test_tensor))
            query = test_tensor[start:end]

            sim = query @ train_tensor.T
            topv, topi = torch.topk(sim, k=n_neighbors, dim=1)

            topv = topv.cpu().numpy()
            topi = topi.cpu().numpy()

            for bi in range(topi.shape[0]):
                labels_i = train_label[topi[bi]]
                scores_i = topv[bi]
                np.maximum.at(out[start + bi], labels_i, scores_i)

    del train_tensor, test_tensor
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


def knn_both_feat(
    n_class,
    train_feat1,
    train_feat2,
    test_feat1,
    test_feat2,
    train_label,
):
    """
    Run KNN with original-image embeddings and flipped-image embeddings.
    Average the two results.
    """
    train_feat12 = np.concatenate([train_feat1, train_feat2], axis=0)
    train_label12 = np.concatenate([train_label, train_label], axis=0)

    knn1 = knn_all_pred_torch(
        n_class=n_class,
        test_feat=test_feat1,
        train_feat=train_feat12,
        train_label=train_label12,
        n_neighbors=CFG.knn_neighbors,
        chunk_size=CFG.knn_chunk_size,
    )

    knn2 = knn_all_pred_torch(
        n_class=n_class,
        test_feat=test_feat2,
        train_feat=train_feat12,
        train_label=train_label12,
        n_neighbors=CFG.knn_neighbors,
        chunk_size=CFG.knn_chunk_size,
    )

    return (knn1 + knn2) / 2.0

In [ ]:
# ============================================================
# Single prototype score matrix
# ============================================================


def build_class_prototypes_from_two_feats(
    train_feat1,
    train_feat2,
    train_label,
    n_class,
):
    """
    Build one averaged embedding prototype for each individual_id.
    """
    feat = np.concatenate([train_feat1, train_feat2], axis=0).astype("float32")
    labels = np.concatenate([train_label, train_label], axis=0)

    feat = normalize_np(feat)

    dim = feat.shape[1]
    prototypes = np.zeros((n_class, dim), dtype=np.float32)
    counts = np.zeros(n_class, dtype=np.int32)

    for f, lab in zip(feat, labels):
        lab = int(lab)
        prototypes[lab] += f
        counts[lab] += 1

    valid_mask = counts > 0

    prototypes[valid_mask] /= counts[valid_mask, None]
    prototypes[valid_mask] = normalize_np(prototypes[valid_mask])

    return prototypes, valid_mask


def prototype_all_pred_torch(
    n_class,
    test_feat1,
    test_feat2,
    train_feat1,
    train_feat2,
    train_label,
    chunk_size=128,
):
    """
    Compute cosine similarity between test embeddings and each individual prototype.

    Returns:
        proto_mat: [N_test, n_class]
    """
    prototypes, valid_mask = build_class_prototypes_from_two_feats(
        train_feat1=train_feat1,
        train_feat2=train_feat2,
        train_label=train_label,
        n_class=n_class,
    )

    test_feat1 = normalize_np(test_feat1.astype("float32"))
    test_feat2 = normalize_np(test_feat2.astype("float32"))

    proto_tensor = torch.tensor(
        prototypes,
        dtype=torch.float32,
    ).to(CFG.device)

    out1 = np.zeros((len(test_feat1), n_class), dtype=np.float32)
    out2 = np.zeros((len(test_feat2), n_class), dtype=np.float32)

    with torch.no_grad():
        test_tensor1 = torch.tensor(
            test_feat1,
            dtype=torch.float32,
        ).to(CFG.device)

        for start in tqdm(
            range(0, len(test_tensor1), chunk_size),
            desc="Prototype pred feat1",
        ):
            end = min(start + chunk_size, len(test_tensor1))
            sim = test_tensor1[start:end] @ proto_tensor.T
            out1[start:end] = sim.cpu().numpy().astype("float32")

        del test_tensor1
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        test_tensor2 = torch.tensor(
            test_feat2,
            dtype=torch.float32,
        ).to(CFG.device)

        for start in tqdm(
            range(0, len(test_tensor2), chunk_size),
            desc="Prototype pred feat2",
        ):
            end = min(start + chunk_size, len(test_tensor2))
            sim = test_tensor2[start:end] @ proto_tensor.T
            out2[start:end] = sim.cpu().numpy().astype("float32")

        del test_tensor2, proto_tensor
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    proto_mat = (out1 + out2) / 2.0

    # Do not assign scores to classes without train-gallery samples.
    proto_mat[:, ~valid_mask] = -1e9

    return proto_mat

In [ ]:
# ============================================================
# Multi-prototype score matrix
# ============================================================


from sklearn.cluster import KMeans


def build_multi_prototypes_from_two_feats(
    train_feat1,
    train_feat2,
    train_label,
    n_class,
    max_centers=3,
    min_samples=4,
    random_state=42,
):
    """
    Build multiple prototypes for each individual_id.

    Returns:
        proto_features: [N_proto, dim]
        proto_labels:   [N_proto]
    """
    feat = np.concatenate([train_feat1, train_feat2], axis=0).astype("float32")
    labels = np.concatenate([train_label, train_label], axis=0)

    feat = normalize_np(feat)

    proto_features = []
    proto_labels = []

    for cls in tqdm(range(n_class), desc="Build multi prototypes"):
        idx = np.where(labels == cls)[0]

        if len(idx) == 0:
            continue

        cls_feat = feat[idx]

        # Few samples: average them into one prototype.
        if len(idx) < min_samples:
            proto = cls_feat.mean(axis=0)
            proto = proto / (np.linalg.norm(proto) + 1e-12)

            proto_features.append(proto.astype("float32"))
            proto_labels.append(cls)
            continue

        # Many samples: split them into at most max_centers centers.
        n_centers = min(max_centers, len(idx))

        try:
            kmeans = KMeans(
                n_clusters=n_centers,
                random_state=random_state,
                n_init=10,
            )

            cluster_ids = kmeans.fit_predict(cls_feat)

            for c in range(n_centers):
                sub_feat = cls_feat[cluster_ids == c]

                if len(sub_feat) == 0:
                    continue

                proto = sub_feat.mean(axis=0)
                proto = proto / (np.linalg.norm(proto) + 1e-12)

                proto_features.append(proto.astype("float32"))
                proto_labels.append(cls)

        except Exception:
            # If KMeans fails, fall back to a single averaged prototype.
            proto = cls_feat.mean(axis=0)
            proto = proto / (np.linalg.norm(proto) + 1e-12)

            proto_features.append(proto.astype("float32"))
            proto_labels.append(cls)

    proto_features = np.stack(proto_features).astype("float32")
    proto_labels = np.array(proto_labels, dtype=np.int64)

    print("Multi prototypes:", proto_features.shape)
    print("Unique prototype labels:", len(np.unique(proto_labels)))

    return proto_features, proto_labels


def multi_prototype_all_pred_torch(
    n_class,
    test_feat1,
    test_feat2,
    train_feat1,
    train_feat2,
    train_label,
    max_centers=3,
    min_samples=4,
    chunk_size=128,
):
    """
    Compute cosine similarity between test embeddings and all prototypes.
    When an individual_id has multiple prototypes, use the maximum similarity.

    Returns:
        proto_mat: [N_test, n_class]
    """
    proto_features, proto_labels = build_multi_prototypes_from_two_feats(
        train_feat1=train_feat1,
        train_feat2=train_feat2,
        train_label=train_label,
        n_class=n_class,
        max_centers=max_centers,
        min_samples=min_samples,
        random_state=CFG.seed if hasattr(CFG, "seed") else 42,
    )

    test_feat1 = normalize_np(test_feat1.astype("float32"))
    test_feat2 = normalize_np(test_feat2.astype("float32"))

    proto_tensor = torch.tensor(
        proto_features,
        dtype=torch.float32,
    ).to(CFG.device)

    def calc_one(test_feat, desc):
        out = np.full((len(test_feat), n_class), -1e9, dtype=np.float32)
        test_tensor = torch.tensor(
            test_feat,
            dtype=torch.float32,
        ).to(CFG.device)

        with torch.no_grad():
            for start in tqdm(
                range(0, len(test_tensor), chunk_size), desc=desc
            ):
                end = min(start + chunk_size, len(test_tensor))
                sim = test_tensor[start:end] @ proto_tensor.T
                sim_np = sim.cpu().numpy().astype("float32")

                # Use the maximum score across prototypes from the same class.
                for j, cls in enumerate(proto_labels):
                    out[start:end, cls] = np.maximum(
                        out[start:end, cls],
                        sim_np[:, j],
                    )

        del test_tensor
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return out

    out1 = calc_one(test_feat1, "Multi-prototype pred feat1")
    out2 = calc_one(test_feat2, "Multi-prototype pred feat2")

    del proto_tensor
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    proto_mat = (out1 + out2) / 2.0

    return proto_mat

In [ ]:
# ============================================================
# Species bonus
# ============================================================


def build_class_species_map(n_class, train_label, train_species_label):
    """
    Build the species label mapping for each individual class.

    class_species[c] = species label of individual class c
    """
    class_species = np.full(n_class, -1, dtype=np.int32)

    for cls, sp in zip(train_label, train_species_label):
        cls = int(cls)
        sp = int(sp)

        if class_species[cls] == -1:
            class_species[cls] = sp

    return class_species


def apply_species_bonus(mat, train_results, test_results, n_class):
    """
    Use species-head predictions to adjust the individual score matrix.

    Current implementation:
    - Add a bonus to individual classes matching the predicted species.
    - CFG.species_penalty is currently recorded but not applied.
    """
    if "label_species" not in train_results:
        raise KeyError(
            "label_species is missing from train_results."
            "Please rerun the updated extract_champion_outputs_for_mode with do_embedding=True."
        )

    if "pred_species_idx" not in test_results:
        raise KeyError(
            "pred_species_idx is missing from test_results."
            "Please rerun the updated extract_champion_outputs_for_mode with do_embedding=True."
        )

    print("Applying species bonus...")

    train_label = train_results["label"]
    train_species_label = train_results["label_species"]

    class_species = build_class_species_map(
        n_class=n_class,
        train_label=train_label,
        train_species_label=train_species_label,
    )

    species_to_classes = {}
    valid_classes = np.where(class_species >= 0)[0]

    for cls_idx in valid_classes:
        sp = int(class_species[cls_idx])

        if sp not in species_to_classes:
            species_to_classes[sp] = []

        species_to_classes[sp].append(cls_idx)

    for sp in species_to_classes:
        species_to_classes[sp] = np.array(
            species_to_classes[sp],
            dtype=np.int64,
        )

    out = mat.astype("float32", copy=True)

    pred_species_idx = test_results["pred_species_idx"]
    use_topk = min(CFG.species_use_topk, pred_species_idx.shape[1])

    for i in tqdm(range(out.shape[0]), desc="Apply species bonus"):
        species_candidates = pred_species_idx[i, :use_topk]

        for rank, sp in enumerate(species_candidates):
            sp = int(sp)

            if sp not in species_to_classes:
                continue

            cls_indices = species_to_classes[sp]
            bonus = CFG.species_bonus / (rank + 1)

            out[i, cls_indices] += bonus

    print(
        f"Species bonus done. "
        f"use_topk={use_topk}, "
        f"bonus={CFG.species_bonus}, "
        f"penalty={CFG.species_penalty} "
        f"(penalty not applied)"
    )

    return out

In [ ]:
# ============================================================
# Threshold search and crop submission
# ============================================================


def binary_search_threshold(mat, new_ratio):
    """
    Find the new_individual threshold.
    Make the ratio of top-1 new_individual predictions close to CFG.new_ratio.
    """
    mat_t = torch.tensor(mat, dtype=torch.float32)

    ok, ng = 0.0, 1.0

    for _ in range(30):
        mid = (ok + ng) / 2

        out_new = torch.cat(
            [
                mat_t,
                torch.full((mat_t.shape[0], 1), mid),
            ],
            dim=1,
        )

        ratio = (out_new.argmax(1) == mat_t.shape[1]).float().mean().item()

        if ratio <= new_ratio:
            ok = mid
        else:
            ng = mid

    return ok


def make_crop_submission_from_matrix(
    mat, test_images, out_prefix, out_dir, summary_rows
):
    """
    Given a score matrix, generate one submission for each new_ratio in CFG.new_ratios.
    """
    os.makedirs(out_dir, exist_ok=True)

    label_classes = np.load(
        os.path.join(CFG.output_dir, "label_classes.npy"),
        allow_pickle=True,
    )

    for new_ratio in CFG.new_ratios:
        threshold = binary_search_threshold(mat, new_ratio)
        print(
            f"{out_prefix} | new_ratio={new_ratio}, selected threshold={threshold}"
        )

        out_new = torch.cat(
            [
                torch.tensor(mat, dtype=torch.float32),
                torch.full((mat.shape[0], 1), threshold),
            ],
            dim=1,
        )

        top5 = out_new.topk(5)[1].numpy()

        def make_pred_str(indices):
            preds = []

            for x in indices:
                if x == mat.shape[1]:
                    preds.append("new_individual")
                else:
                    preds.append(str(label_classes[x]))

            return " ".join(preds)

        submission = pd.DataFrame(
            {
                "image": test_images,
                "predictions": [make_pred_str(ids) for ids in top5],
            }
        )

        out_path = os.path.join(
            out_dir,
            f"{out_prefix}_newratio{new_ratio:.3f}_th{threshold:.6f}.csv",
        )

        submission.to_csv(out_path, index=False)

        print("Saved:", out_path)
        print(submission.head())

        summary_rows.append(
            {
                "out_prefix": out_prefix,
                "new_ratio": new_ratio,
                "threshold": threshold,
                "out_path": out_path,
            }
        )

In [ ]:
# ============================================================
# Component Mat Builder
# ============================================================


def build_or_load_component_mats_for_crop(crop_mode, component_dir):
    """
    Build or load component matrices for one crop mode.

    This step only builds:
    - knn.npy
    - logit.npy
    - proto_single.npy
    - proto_multi.npy
    - test_images.npy

    This step does not do:
    - weighted fusion of KNN/logit/prototype scores
    - species bonus
    - crop ensemble
    - submission
    """
    os.makedirs(component_dir, exist_ok=True)

    crop_dir = os.path.join(component_dir, crop_mode)
    os.makedirs(crop_dir, exist_ok=True)

    paths = {
        "knn": os.path.join(crop_dir, "knn.npy"),
        "logit": os.path.join(crop_dir, "logit.npy"),
        "proto_single": os.path.join(crop_dir, "proto_single.npy"),
        "proto_multi": os.path.join(crop_dir, "proto_multi.npy"),
        "test_images": os.path.join(crop_dir, "test_images.npy"),
    }

    train_path = os.path.join(CFG.output_dir, f"train_{crop_mode}_results.npz")
    test_path = os.path.join(CFG.output_dir, f"test_{crop_mode}_results.npz")

    if not os.path.exists(train_path):
        raise FileNotFoundError(
            f"Missing train result: {train_path}\n"
            f"Please make sure Hugging Face embeddings are downloaded, or run with CFG.do_embedding=True first."
        )

    if not os.path.exists(test_path):
        raise FileNotFoundError(
            f"Missing test result: {test_path}\n"
            f"Please make sure Hugging Face embeddings are downloaded, or run with CFG.do_embedding=True first."
        )

    print("\n" + "=" * 80)
    print(f"Build / load component mats for crop mode: {crop_mode}")
    print("=" * 80)
    print("train_path:", train_path)
    print("test_path:", test_path)

    train_results = np.load(train_path, allow_pickle=True)
    test_results = np.load(test_path, allow_pickle=True)

    train_label = train_results["label"]
    test_images = test_results["image"]

    # Save / check test image order
    if not os.path.exists(paths["test_images"]):
        np.save(paths["test_images"], test_images)
        print("Saved test_images:", paths["test_images"])
    else:
        cached_test_images = np.load(paths["test_images"], allow_pickle=True)
        assert np.array_equal(cached_test_images, test_images), (
            f"Test image order changed for {crop_mode}. "
            f"Please delete the old component-matrix cache and rebuild it."
        )
        print(
            "test_images cache exists and order matches:", paths["test_images"]
        )

    # ------------------------------------------------------------
    # 1. KNN mat
    # ------------------------------------------------------------
    if getattr(CFG, "build_knn_mat", True):
        if os.path.exists(paths["knn"]):
            print("KNN mat already exists:", paths["knn"])
        else:
            print("\nBuilding KNN mat:", crop_mode)

            knn_mat = knn_both_feat(
                num_classes,
                train_results["embed_features1"],
                train_results["embed_features2"],
                test_results["embed_features1"],
                test_results["embed_features2"],
                train_label,
            )

            np.save(paths["knn"], knn_mat.astype("float32"))
            print("Saved KNN mat:", paths["knn"])

            del knn_mat
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # ------------------------------------------------------------
    # 2. Logit mat
    # ------------------------------------------------------------
    if getattr(CFG, "build_logit_mat", True):
        if os.path.exists(paths["logit"]):
            print("Logit mat already exists:", paths["logit"])
        else:
            print("\nBuilding logit mat:", crop_mode)

            logit_mat = restore_all_pred(
                num_classes,
                test_results["pred_logit"],
                test_results["pred_idx"],
            )

            np.save(paths["logit"], logit_mat.astype("float32"))
            print("Saved logit mat:", paths["logit"])

            del logit_mat
            gc.collect()

    # ------------------------------------------------------------
    # 3. Single prototype mat
    # ------------------------------------------------------------
    if getattr(CFG, "build_proto_single_mat", True):
        if os.path.exists(paths["proto_single"]):
            print("Single proto mat already exists:", paths["proto_single"])
        else:
            print("\nBuilding single prototype mat:", crop_mode)

            proto_single_mat = prototype_all_pred_torch(
                n_class=num_classes,
                test_feat1=test_results["embed_features1"],
                test_feat2=test_results["embed_features2"],
                train_feat1=train_results["embed_features1"],
                train_feat2=train_results["embed_features2"],
                train_label=train_label,
                chunk_size=CFG.knn_chunk_size,
            )

            np.save(paths["proto_single"], proto_single_mat.astype("float32"))
            print("Saved single proto mat:", paths["proto_single"])

            del proto_single_mat
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # ------------------------------------------------------------
    # 4. Multi prototype mat
    # ------------------------------------------------------------
    if getattr(CFG, "build_proto_multi_mat", True):
        if os.path.exists(paths["proto_multi"]):
            print("Multi proto mat already exists:", paths["proto_multi"])
        else:
            print("\nBuilding multi prototype mat:", crop_mode)

            proto_multi_mat = multi_prototype_all_pred_torch(
                n_class=num_classes,
                test_feat1=test_results["embed_features1"],
                test_feat2=test_results["embed_features2"],
                train_feat1=train_results["embed_features1"],
                train_feat2=train_results["embed_features2"],
                train_label=train_label,
                max_centers=CFG.multi_proto_max_centers,
                min_samples=CFG.multi_proto_min_samples,
                chunk_size=CFG.knn_chunk_size,
            )

            np.save(paths["proto_multi"], proto_multi_mat.astype("float32"))
            print("Saved multi proto mat:", paths["proto_multi"])

            del proto_multi_mat
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    del train_results, test_results
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return paths

In [ ]:
# ============================================================
# Fuse component mats for one crop
# ============================================================


def load_and_fuse_crop_mat(crop_mode, component_dir):
    """
    Load component matrices for one crop mode and generate a fused crop matrix using the current CFG.

    Changing these settings does not require rebuilding component matrices:
    - CFG.knn_weight
    - CFG.logit_weight
    - CFG.proto_weight
    - CFG.proto_mode
    - CFG.knn_ratio
    """
    crop_dir = os.path.join(component_dir, crop_mode)

    knn_path = os.path.join(crop_dir, "knn.npy")
    logit_path = os.path.join(crop_dir, "logit.npy")
    proto_single_path = os.path.join(crop_dir, "proto_single.npy")
    proto_multi_path = os.path.join(crop_dir, "proto_multi.npy")
    test_images_path = os.path.join(crop_dir, "test_images.npy")

    required_paths = [knn_path, logit_path, test_images_path]
    for p in required_paths:
        if not os.path.exists(p):
            raise FileNotFoundError(
                f"Missing component mat: {p}\n"
                f"Please run CFG.run_mode = 'build_component_mats' first."
            )

    print("\n" + "=" * 80)
    print(f"Fuse crop mat: {crop_mode}")
    print("=" * 80)

    knn_mat = np.load(knn_path, mmap_mode="r")
    logit_mat = np.load(logit_path, mmap_mode="r")
    test_images = np.load(test_images_path, allow_pickle=True)

    if getattr(CFG, "use_proto_score", False):
        if CFG.proto_mode == "single":
            proto_path = proto_single_path
        elif CFG.proto_mode == "multi":
            proto_path = proto_multi_path
        else:
            raise ValueError(f"Unknown CFG.proto_mode: {CFG.proto_mode}")

        if not os.path.exists(proto_path):
            raise FileNotFoundError(
                f"Missing proto mat: {proto_path}\n"
                f"Please run CFG.run_mode = 'build_component_mats' first."
            )

        proto_mat = np.load(proto_path, mmap_mode="r")

        total_w = CFG.knn_weight + CFG.logit_weight + CFG.proto_weight
        if total_w <= 0:
            raise ValueError(
                "knn_weight + logit_weight + proto_weight must be > 0"
            )

        print("Using proto score")
        print("proto_mode:", CFG.proto_mode)
        print("knn_weight:", CFG.knn_weight)
        print("logit_weight:", CFG.logit_weight)
        print("proto_weight:", CFG.proto_weight)
        print(
            "normalized weights:",
            CFG.knn_weight / total_w,
            CFG.logit_weight / total_w,
            CFG.proto_weight / total_w,
        )

        fused = (
            (CFG.knn_weight / total_w) * knn_mat
            + (CFG.logit_weight / total_w) * logit_mat
            + (CFG.proto_weight / total_w) * proto_mat
        ).astype("float32")

    else:
        print("No proto score")
        print("knn_ratio:", CFG.knn_ratio)

        fused = (
            CFG.knn_ratio * knn_mat + (1.0 - CFG.knn_ratio) * logit_mat
        ).astype("float32")

    return fused, test_images

In [ ]:
# ============================================================
# Apply species bonus for one crop
# ============================================================


def apply_species_bonus_for_crop_if_needed(mat, crop_mode):
    """
    If CFG.use_species_bonus=True, load the train/test npz files for this crop mode.
    Apply the species bonus to the fused matrix.
    """
    if not getattr(CFG, "use_species_bonus", False):
        return mat

    train_path = os.path.join(CFG.output_dir, f"train_{crop_mode}_results.npz")
    test_path = os.path.join(CFG.output_dir, f"test_{crop_mode}_results.npz")

    if not os.path.exists(train_path):
        raise FileNotFoundError(f"Missing train result: {train_path}")

    if not os.path.exists(test_path):
        raise FileNotFoundError(f"Missing test result: {test_path}")

    train_results = np.load(train_path, allow_pickle=True)
    test_results = np.load(test_path, allow_pickle=True)

    mat = apply_species_bonus(
        mat=mat,
        train_results=train_results,
        test_results=test_results,
        n_class=num_classes,
    )

    del train_results, test_results
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return mat

In [ ]:
# ============================================================
# Build all component mats
# ============================================================


def build_all_component_mats():
    """
    Only build component matrices; do not generate submissions.

    This step is slower, but it does not need to be rerun as long as embeddings stay unchanged.
    """
    print("\n" + "=" * 80)
    print("RUN: BUILD COMPONENT MATS")
    print("=" * 80)

    component_dir = os.path.join(CFG.output_dir, CFG.component_mat_dir)
    os.makedirs(component_dir, exist_ok=True)

    for crop_mode in CFG.crop_infer_modes:
        print("\n" + "-" * 80)
        print(f"Crop mode: {crop_mode}")
        print("-" * 80)

        # Download embeddings from Hugging Face when needed.
        download_embeddings_for_mode(crop_mode)

        build_or_load_component_mats_for_crop(
            crop_mode=crop_mode,
            component_dir=component_dir,
        )

    # Optional: upload component matrices.
    hf_upload_folder_if_exists(
        component_dir,
        repo_path=f"{CFG.hf_mat_dir}/{CFG.component_mat_dir}",
        commit_message="Upload component mats",
    )

    print("\nDone building component mats.")
    print("Component dir:", component_dir)

In [ ]:
# ============================================================
# Generate submissions from component mats
# ============================================================


def generate_crop_submissions_from_component_mats():
    """
    Only read component matrices and generate submissions using the current CFG weights.

    After changing these settings, only rerun this step:
    - knn_weight / logit_weight / proto_weight
    - proto_mode
    - knn_ratio
    - crop_weight_sets
    - new_ratios
    - use_species_bonus / species_bonus
    """
    print("\n" + "=" * 80)
    print("RUN: GENERATE SUBMISSIONS FROM COMPONENT MATS")
    print("=" * 80)

    component_dir = os.path.join(CFG.output_dir, CFG.component_mat_dir)
    out_dir = os.path.join(CFG.output_dir, CFG.crop_submission_dir)

    os.makedirs(out_dir, exist_ok=True)

    summary_rows = []
    test_images_ref = None

    # Verify the test image order first.
    for crop_mode in CFG.crop_infer_modes:
        test_images_path = os.path.join(
            component_dir,
            crop_mode,
            "test_images.npy",
        )

        if not os.path.exists(test_images_path):
            raise FileNotFoundError(
                f"Missing test_images cache: {test_images_path}\n"
                f"Please run CFG.run_mode = 'build_component_mats' first."
            )

        test_images = np.load(test_images_path, allow_pickle=True)

        if test_images_ref is None:
            test_images_ref = test_images
        else:
            assert np.array_equal(test_images_ref, test_images), (
                f"Test image order mismatch for {crop_mode}"
            )

    # ------------------------------------------------------------
    # Weighted crop ensemble
    # ------------------------------------------------------------
    for item in CFG.crop_weight_sets:
        name = item["name"]
        weights = item["weights"]

        print("\n" + "=" * 80)
        print("Weighted ensemble:", name)
        print("=" * 80)
        print(weights)

        total_weight = sum(
            weights.get(crop_mode, 0.0) for crop_mode in CFG.crop_infer_modes
        )

        assert abs(total_weight - 1.0) < 1e-6, (
            f"Weights should sum to 1 over {CFG.crop_infer_modes}, "
            f"got {total_weight}"
        )

        ensemble_mat = None

        for crop_mode in CFG.crop_infer_modes:
            w = weights.get(crop_mode, 0.0)
            print(f"  {crop_mode}: {w}")

            if w == 0:
                continue

            crop_mat, _ = load_and_fuse_crop_mat(
                crop_mode=crop_mode,
                component_dir=component_dir,
            )

            crop_mat = apply_species_bonus_for_crop_if_needed(
                mat=crop_mat,
                crop_mode=crop_mode,
            )

            if ensemble_mat is None:
                ensemble_mat = crop_mat.astype("float32") * w
            else:
                ensemble_mat += crop_mat.astype("float32") * w

            del crop_mat
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        prefix = f"weighted_{name}_{CFG.proto_tag}_{CFG.score_tag}"

        if getattr(CFG, "use_species_bonus", False):
            prefix += "_species"

        make_crop_submission_from_matrix(
            mat=ensemble_mat,
            test_images=test_images_ref,
            out_prefix=prefix,
            out_dir=out_dir,
            summary_rows=summary_rows,
        )

        del ensemble_mat
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ------------------------------------------------------------
    # Save summary
    # ------------------------------------------------------------
    summary_df = pd.DataFrame(summary_rows)
    summary_path = os.path.join(out_dir, "submission_summary.csv")
    summary_df.to_csv(summary_path, index=False)

    print("\nSaved summary:", summary_path)
    print(summary_df)

    # ------------------------------------------------------------
    # Upload submissions
    # ------------------------------------------------------------
    hf_upload_folder_if_exists(
        out_dir,
        repo_path=CFG.crop_submission_dir,
        commit_message=f"Upload {CFG.crop_submission_dir}",
    )

    print("\nDone generating submissions.")
    print("Output dir:", out_dir)

In [ ]:
# ============================================================
# Run inference
# ============================================================


CFG.run_mode = "crop_inference"

if CFG.run_mode == "build_component_mats":
    build_all_component_mats()

elif CFG.run_mode == "make_submissions":
    generate_crop_submissions_from_component_mats()

elif CFG.run_mode == "crop_inference":
    # First full run: build component matrices, then generate submissions.
    build_all_component_mats()
    generate_crop_submissions_from_component_mats()

else:
    print("Skip inference. CFG.run_mode =", CFG.run_mode)

# Shutdown Colab Runtime

In [ ]:
# if CFG.auto_disconnect_when_done and IN_COLAB:
#     print("Completed. Auto-disconnecting Colab runtime.")
#     try:
#         runtime.unassign()
#     except Exception as e:
#         print("Auto disconnect failed. Please disconnect manually.")
#         print("Error:", e)